# Lesson 08｜JSON API、開放資料與登入

前六堂我們都在解析 HTML。但很多網頁的資料其實是從「API」拿到 JSON 再畫出來的：
找到這個來源，直接取用，比解析 HTML 更簡單也更穩。
今天分三段：

- **A. JSON 與 API**：用 F12 找資料來源，把 JSON 變成 DataFrame
- **B. 開放資料**：用台北市 YouBike 即時資料做一次分析
- **C. 登入與 Cookies**：用 `requests.Session` 登入本機練習站，抓會員才看得到的頁面

在 VS Code 按每格上方的 Run Cell 逐格執行。


### Notebook 使用方式

先選擇課程 Python 環境，再由上到下執行每一格（Shift + Enter）。先執行工作資料夾設定及前面的匯入。
若看到 `NameError`，先確認變數名稱，並回到上方執行建立該變數的格子；必要時重新啟動核心並從頭執行。練習或作業格要先完成 TODO，再執行檢查。


## 設定工作資料夾

請先執行下一格。使用 VS Code 開啟完整教材資料夾；本堂的 `fixtures/` 和 `output/` 會以本堂資料夾為基準。


In [ ]:
import os
from pathlib import Path

# 從教材根目錄或本堂資料夾啟動 Notebook 都可以。
current = Path.cwd().resolve()
lesson_name = 'lesson08_json_api_and_sessions'
candidates = [current, *current.parents]
lesson_path = next((p for p in candidates if p.name == lesson_name and (p / 'fixtures').is_dir()), None)
if lesson_path is None:
    lesson_path = next((p / 'lessons' / lesson_name for p in candidates if (p / 'lessons' / lesson_name / 'fixtures').is_dir()), None)
if lesson_path is None:
    raise FileNotFoundError('找不到本堂教材。請用 VS Code 開啟完整教材資料夾，重新啟動核心後再執行。')
os.chdir(lesson_path)
print('工作資料夾：', Path.cwd())

import sys
tools_path = str(lesson_path.parents[1] / 'tools')
if tools_path not in sys.path:
    sys.path.insert(0, tools_path)


In [ ]:
import json
from io import StringIO

import pandas as pd
import requests
from bs4 import BeautifulSoup

USER_AGENT = "python-scraping-viz-class/0.1 (student project)"


def fetch(url, params=None, timeout=10):
    headers = {"User-Agent": USER_AGENT}
    try:
        response = requests.get(url, params=params, headers=headers, timeout=timeout)
        response.raise_for_status()
    except requests.RequestException as error:
        print(f"抓取失敗：{url} → {type(error).__name__}")
        return None
    return response


## 01｜今天的成果預覽

同樣是名言：不用 BeautifulSoup、不用選擇器，JSON 直接變成表格。
這裡先讀 `fixtures/` 裡存好的 API 回應。


In [ ]:
with open("fixtures/quotes_api_page1.json", encoding="utf-8") as file:
    page1 = json.load(file)

preview = pd.json_normalize(page1["quotes"], sep="_")
print(preview[["author_name", "text"]].head(3).to_string(index=False))


# A｜JSON 與 API


## 02｜JSON 是什麼

JSON 是一種文字格式，長得和 Python 的 dict、list 幾乎一樣。

| JSON | Python |
|---|---|
| `{...}` 物件 | `dict` |
| `[...]` 陣列 | `list` |
| 字串、數字 | `str`、`int`、`float` |
| `true`、`false` | `True`、`False` |
| `null` | `None` |

`json.loads` 把 JSON 文字變成 Python 物件；`json.dumps` 反過來。


In [ ]:
text = '{"name": "小明", "scores": [90, 85], "address": {"city": "台北"}, "active": true, "nickname": null}'
data = json.loads(text)

print(type(data).__name__, data["name"], data["scores"][0])
print("巢狀：", data["address"]["city"])
print("true → ", data["active"], "｜null → ", data["nickname"])


In [ ]:
# ensure_ascii=False 讓中文照原樣顯示；indent 讓結構好讀
print(json.dumps(data, ensure_ascii=False, indent=2))


## 03｜用 F12 找到資料真正的來源

名言網站有一頁會一直往下捲動載入：https://quotes.toscrape.com/scroll

1. 用 Chrome 打開這頁，按 F12，切到 **Network**，再點 **Fetch/XHR**。
2. 往下捲動，會看到新的請求 `quotes?page=2`。
3. 點它，切到 **Response**：就是 JSON。網址就是資料的來源（API）。

這和第 5 堂的翻頁不一樣：頁面上根本沒有「下一頁」連結，資料是 JavaScript 去 API 拿的。
只要找到這個 API，用 `requests` 就能拿到，不需要瀏覽器。


## 04｜呼叫 API

`response.json()` 會把回應直接轉成 Python 物件。`params` 會組成網址的查詢參數。


In [ ]:
response = fetch("https://quotes.toscrape.com/api/quotes", params={"page": 1})
if response is None:
    print("沒有網路，改用本機快照")
    data = page1
else:
    data = response.json()
    print("實際呼叫的網址：", response.url)
    print("Content-Type：", response.headers["Content-Type"])

print("頁碼：", data["page"], "｜還有下一頁：", data["has_next"], "｜本頁名言數：", len(data["quotes"]))


## 05｜看懂 JSON 的結構

拿到陌生的 JSON，先看三件事：最外層是 dict 還是 list、有哪些 key、資料在哪一層。


In [ ]:
print("最外層 key：", list(data))
print("quotes 的型態：", type(data["quotes"]).__name__, "，第一筆的 key：", list(data["quotes"][0]))


In [ ]:
print(json.dumps(data["quotes"][0], ensure_ascii=False, indent=2))


## 06｜取出巢狀的值

一層一層用 `[...]` 往下取。key 可能不存在時，用 `.get()`，找不到回傳 `None` 而不是報錯。


In [ ]:
first = data["quotes"][0]
print("作者：", first["author"]["name"])
print("標籤：", first["tags"])
print("不存在的欄位：", first.get("rating"))
print("安全取巢狀值：", first.get("author", {}).get("name"))


## 07｜API 翻頁

這個 API 用 `has_next` 告訴你有沒有下一頁。邏輯和第 5 堂的翻頁迴圈一樣，但不需要解析 HTML、也不需要 `urljoin`。


In [ ]:
import time


def fetch_quotes(max_pages=3, delay=1.0):
    quotes = []
    page = 1
    while page <= max_pages:
        response = fetch("https://quotes.toscrape.com/api/quotes", params={"page": page})
        if response is None:
            break
        payload = response.json()
        quotes.extend(payload["quotes"])
        print(f"第 {page} 頁：{len(payload['quotes'])} 則（累計 {len(quotes)}）")
        if not payload["has_next"]:
            break
        page += 1
        time.sleep(delay)
    return quotes


all_quotes = fetch_quotes(max_pages=3)
print("共", len(all_quotes), "則")


## 08｜json_normalize：巢狀 JSON → 平面的表格

`author` 是巢狀的 dict。`pd.json_normalize` 會把它展開成 `author_name`、`author_slug` 這種欄位。


In [ ]:
df = pd.json_normalize(all_quotes or page1["quotes"], sep="_")
print(df.columns.tolist())
print(df[["author_name", "author_slug"]].head(3).to_string(index=False))
print("tags 還是 list：", df["tags"].iloc[0])


## 09｜API 與 HTML 爬蟲，怎麼選？

| | 解析 HTML | 呼叫 API |
|---|---|---|
| 程式 | 要寫選擇器、處理缺值 | `response.json()` 就是資料 |
| 穩定度 | 網頁改版就壞 | 格式較固定 |
| 資料 | 只有畫面上看得到的 | 常有更多欄位 |
| 注意 | robots.txt、服務條款 | 授權、金鑰、呼叫頻率 |

**有 API 或開放資料，就不要爬網頁**，這是第 2 堂爬取守則的第一條。


# B｜開放資料


## 10｜什麼是開放資料

政府與機構把資料公開，授權大家使用。找資料的入口是[政府資料開放平臺](https://data.gov.tw/)。
使用前先看三件事：

1. **授權**：多數政府資料適用「政府資料開放授權條款」，可以重製、改作、散布，要標示出處。
2. **格式**：JSON、CSV、XML；有的要申請金鑰，有的不用。
3. **更新頻率**：即時資料每分鐘更新，數字每次不一樣。

今天用台北市 YouBike 2.0 的即時資料（不用金鑰）。全部約 1800 站、近 1MB，
`fixtures/` 裡存了每 6 站取 1 站的快照（302 站，涵蓋所有行政區），課堂分析用快照，結果才會一致。


## 11｜呼叫即時資料（示範）


In [ ]:
YOUBIKE_URL = "https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json"

live = fetch(YOUBIKE_URL, timeout=20)
if live is not None:
    stations_live = live.json()
    print("即時資料站數：", len(stations_live))
    print("第一站：", stations_live[0]["sna"], "｜可借車輛：", stations_live[0]["available_rent_bikes"])
    print("資料更新時間：", stations_live[0]["srcUpdateTime"])


## 12｜快照 → DataFrame


In [ ]:
with open("fixtures/youbike_sample.json", encoding="utf-8") as file:
    stations = json.load(file)

bike = pd.DataFrame(stations)
print("形狀：", bike.shape)
print(bike.columns.tolist())
print(bike[["sna", "sarea", "Quantity", "available_rent_bikes", "available_return_bikes", "act"]].head(3).to_string(index=False))


欄位名稱要查資料說明才看得懂：`Quantity` 是總車位、`available_rent_bikes` 是可借車輛、
`available_return_bikes` 是可還空位、`act` 是營運狀態（`"1"` 營運中、`"0"` 暫停）。
注意 `act` 是**字串**，不是數字。


In [ ]:
bike = bike.rename(
    columns={
        "sna": "station",
        "sarea": "district",
        "Quantity": "docks",
        "available_rent_bikes": "bikes",
        "available_return_bikes": "free_docks",
    }
)
bike["station"] = bike["station"].str.replace("YouBike2.0_", "", regex=False)
bike["active"] = bike["act"].eq("1")
print(bike[["station", "district", "docks", "bikes", "active"]].head(3).to_string(index=False))
print("營運中：", bike["active"].sum(), "站，暫停：", (~bike["active"]).sum(), "站")


## 13｜分析：哪個行政區的車最多？


In [ ]:
by_district = bike.groupby("district").agg(stations=("station", "count"), bikes=("bikes", "sum"), docks=("docks", "sum"))
by_district["bike_rate"] = (by_district["bikes"] / by_district["docks"]).round(3)
print(by_district.sort_values("bikes", ascending=False).head(5).to_string())


In [ ]:
no_bike = bike[bike["bikes"] == 0]
print("沒有車可借的站：", len(no_bike), "站，占", round(len(no_bike) / len(bike) * 100, 1), "%")
print(no_bike["district"].value_counts().head(3).to_string())


## 14｜JSON 回應出錯時

- 狀態碼不是 200：`fetch` 已經處理。
- 狀態碼 200，但內容不是 JSON（例如被導向錯誤頁）：`response.json()` 會丟出 `JSONDecodeError`。


In [ ]:
def get_json(url, params=None, timeout=10):
    response = fetch(url, params=params, timeout=timeout)
    if response is None:
        return None
    try:
        return response.json()
    except requests.JSONDecodeError:
        print("回應不是 JSON，Content-Type：", response.headers.get("Content-Type"))
        return None


print(get_json("https://quotes.toscrape.com/"))  # 這是 HTML，不是 JSON


## 15｜存檔：JSON 與 CSV


In [ ]:
Path("output").mkdir(exist_ok=True)

with open("output/quotes_raw.json", "w", encoding="utf-8") as file:
    json.dump(all_quotes or page1["quotes"], file, ensure_ascii=False, indent=2)

bike.to_csv("output/youbike_by_station.csv", index=False, encoding="utf-8-sig")
by_district.to_csv("output/youbike_by_district.csv", encoding="utf-8-sig")
print("已存檔：output/quotes_raw.json、youbike_by_station.csv、youbike_by_district.csv")


# C｜登入與 Cookies


## 16｜為什麼登入後才看得到資料

登入的原理：

1. 你送出帳號密碼（POST）。
2. 伺服器確認後，回一個 **Cookie**（一段識別碼，像會員的手環）。
3. 之後每個請求都帶著這個 Cookie，伺服器就知道你是誰。

瀏覽器會自動帶 Cookie。`requests.Session()` 也會，所以登入後用同一個 `session` 發請求就好。

**使用守則**：只登入你自己有帳號、而且服務條款允許的網站；真實的帳號密碼不要寫進程式碼。
今天用的是一個只在你電腦上執行的本機練習站，帳號密碼是測試用的。


## 17｜啟動本機練習站

`start_server()` 在背景啟動練習站，回傳網址。用瀏覽器打開看看：登入頁、會員頁、搜尋頁。


In [ ]:
from practice_site import TEST_PASSWORD, TEST_USER, start_server, stop_server

server, BASE_URL = start_server()
print("本機練習站：", BASE_URL)


In [ ]:
# 還沒登入就去會員頁：伺服器回 302，把你導向登入頁
anonymous = requests.get(f"{BASE_URL}/members", allow_redirects=False, timeout=10)
print("狀態碼：", anonymous.status_code, "｜導向：", anonymous.headers["Location"])


## 18｜看懂登入表單

先用 GET 取得登入頁，看表單有哪些欄位。`csrf_token` 是隱藏欄位，每次載入頁面都不同，
登入時必須一起送出，這是網站防止偽造表單的做法。


In [ ]:
session = requests.Session()
login_page = session.get(f"{BASE_URL}/login", timeout=10)

form = BeautifulSoup(login_page.text, "html.parser").select_one("form#login-form")
print("表單送到：", form["method"].upper(), form["action"])
for field in form.select("input"):
    print(" 欄位：", field.get("name"), "｜type =", field.get("type"))


## 19｜用 Session 登入

把隱藏欄位的值、帳號、密碼組成 `data`，用同一個 `session` 送出 POST。


In [ ]:
token = form.select_one("input[name=csrf_token]")["value"]
login_response = session.post(
    f"{BASE_URL}/login",
    data={"csrf_token": token, "username": TEST_USER, "password": TEST_PASSWORD},
    timeout=10,
)

print("狀態碼：", login_response.status_code)
print("轉址紀錄：", [(step.status_code, step.url.replace(BASE_URL, "")) for step in login_response.history])
print("最後停在：", login_response.url.replace(BASE_URL, ""))
print("session 持有的 cookie 名稱：", list(session.cookies.keys()))


## 20｜抓會員才看得到的頁面

登入後，同一個 `session` 就帶著 Cookie。頁面上的 `<table>` 用第 6 堂的 `read_html` 變成 DataFrame。


In [ ]:
members = session.get(f"{BASE_URL}/members", timeout=10)
print(BeautifulSoup(members.text, "html.parser").select_one("#welcome").get_text())

orders = pd.read_html(StringIO(members.text))[0]
print(orders.to_string(index=False))
print("已付款的金額合計：", orders.loc[orders["狀態"] == "已付款", "金額"].sum())


In [ ]:
# 沒有帶 Cookie 的請求（沒有用 session）：被導回登入頁
without_cookie = requests.get(f"{BASE_URL}/members", timeout=10)
print("沒登入的請求最後停在：", without_cookie.url.replace(BASE_URL, ""))


## 21｜小結與收尾

- 找資料來源：F12 → Network → Fetch/XHR。有 API 就用 API。
- JSON → Python → DataFrame：`response.json()`、`pd.json_normalize`。
- 開放資料：先看授權、格式、更新頻率。
- 要登入：`requests.Session` 自動帶 Cookie；表單的隱藏欄位要一起送。

用完把本機練習站關掉。


In [ ]:
stop_server(server)
print("練習站已關閉")
